# Network and Emotional Structure of Text as a Cognitive Signal

This notebook models short texts as networks instead of bags of words, then tests
whether the shape of that network — how connected it is, how many isolated concepts
it has, how emotionally expressive it is — carries a signal about the writer's
cognitive state.

The texts here are LLM-generated to simulate two writing styles: one resembling
Alzheimer's disease (AD) patients, one resembling age-matched healthy controls (HC).
The pipeline itself — turning text into a graph, testing which graph properties
differ between groups, then checking whether those properties can classify new
text — generalizes to any setting where you want to compare how two groups
structure their language: user research transcripts, support tickets, survey
open-ends, or written feedback grouped by segment.

This version adds a dedicated robustness section: rather than stopping at a good
classification number, it explicitly tests whether that number reflects a genuine
signal or just how the synthetic data was generated.


In [ ]:
# One-time setup. EmoAtlas builds the text network and assigns NRC emotion
# scores; spaCy's large English model and NLTK's WordNet are its dependencies.
# !pip install git+https://github.com/MassimoStel/emoatlas
# !python -m spacy download en_core_web_lg
# import nltk; nltk.download('wordnet'); nltk.download('omw-1.4')


In [ ]:
import json
import re

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from emoatlas import EmoScores
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate, cross_val_score
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
import xgboost as xgb
import shap

emos = EmoScores(language="english")


## 1. Data

Each text is grouped by simulated condition (`alzheimer` / `healthy`) in a JSON file.
A small illustrative sample is included at `data/sample_merged_paragraphs.json` — short,
template-generated texts that mimic the format of the real corpus, used only to keep
this notebook runnable end to end. See `data/README.md` for how the real corpus was
generated and where to place it for a full run. The figures embedded in the README
and in `docs/report.pdf` are from the actual analysis on the full corpus.


In [ ]:
with open("data/sample_merged_paragraphs.json") as f:
    data = json.load(f)

rows = []
for block in data:
    for text in block.get("alzheimer", []):
        rows.append({"text": text, "label": "alzheimer"})
    for text in block.get("healthy", []):
        rows.append({"text": text, "label": "healthy"})

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} documents")
print(df["label"].value_counts())


## 2. From text to network: feature extraction

Each text is converted into a Textual Forma Mentis Network (TFMN) via EmoAtlas, which
links words based on syntactic dependency proximity rather than simple co-occurrence
counts. From the resulting graph we extract two feature families:

- **Topological features**: size and connectivity of the network (nodes, edges,
  density, clustering, number of components, largest-connected-component size,
  average path length, diameter, isolated nodes)
- **Emotional features**: z-scores for the 8 basic emotions in Plutchik's model,
  computed against the NRC Emotion Lexicon baseline


In [ ]:
TOPO_COLS = ["n_nodes", "n_edges", "density", "mean_degree", "clustering",
             "n_components", "lcc_size", "lcc_ratio", "avg_path_length",
             "diameter", "n_isolated"]
EMOTION_COLS = ["anger", "trust", "surprise", "disgust", "joy", "sadness", "fear", "anticipation"]


def extract_network_features(fmnt):
    G = nx.Graph()
    G.add_nodes_from(fmnt.vertices)
    G.add_edges_from(fmnt.edges)

    if G.number_of_nodes() == 0:
        return {}

    components = list(nx.connected_components(G))
    largest_component = G.subgraph(max(components, key=len)).copy()

    return {
        "n_nodes": G.number_of_nodes(),
        "n_edges": G.number_of_edges(),
        "density": nx.density(G),
        "mean_degree": np.mean([d for _, d in G.degree()]),
        "clustering": nx.average_clustering(G),
        "n_components": len(components),
        "lcc_size": len(largest_component),
        "lcc_ratio": len(largest_component) / G.number_of_nodes(),
        "avg_path_length": nx.average_shortest_path_length(largest_component)
                            if len(largest_component) > 1 else 0,
        "diameter": nx.diameter(largest_component) if len(largest_component) > 1 else 0,
        "n_isolated": len(list(nx.isolates(G))),
    }


def extract_features(text):
    fmnt = emos.formamentis_network(text)
    network_features = extract_network_features(fmnt)
    emotion_features = emos.zscores(text)
    return {**network_features, **emotion_features}


In [ ]:
records = []
for _, row in df.iterrows():
    features = extract_features(row["text"])
    features["label"] = row["label"]
    records.append(features)

df_features = pd.DataFrame(records).fillna(0)
print(df_features.shape)
df_features.groupby("label")[TOPO_COLS].median().T.round(3)


## 3. Do the network structures actually differ?

A non-parametric Mann-Whitney U test compares each topological feature between
groups, with Benjamini-Hochberg FDR correction to control for testing 11 features
at once.

![Topological features compared between groups](assets/topological_boxplots.png)

*Boxplots from the actual analysis: AD-simulated networks are consistently smaller,
more fragmented, and structurally simpler than healthy-simulated ones (all differences
significant after FDR correction).*


In [ ]:
alz = df_features[df_features["label"] == "alzheimer"]
hlt = df_features[df_features["label"] == "healthy"]

stat_results = []
for col in TOPO_COLS:
    stat, p = mannwhitneyu(alz[col], hlt[col], alternative="two-sided")
    effect_size_r = 1 - (2 * stat) / (len(alz) * len(hlt))
    stat_results.append({
        "feature": col,
        "alz_median": alz[col].median(),
        "hlt_median": hlt[col].median(),
        "p": p,
        "effect_size_r": round(effect_size_r, 3),
    })

stats_df = pd.DataFrame(stat_results)
_, stats_df["p_adj"], _, _ = multipletests(stats_df["p"], method="fdr_bh")
stats_df["significant"] = stats_df["p_adj"] < 0.05

stats_df.sort_values("p_adj").round(4)


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(TOPO_COLS):
    sns.boxplot(data=df_features, x="label", y=col, ax=axes[i],
                hue="label", palette={"alzheimer": "tomato", "healthy": "steelblue"}, legend=False)
    p_adj = stats_df.loc[stats_df["feature"] == col, "p_adj"].values[0]
    sig = "***" if p_adj < 0.001 else "**" if p_adj < 0.01 else "*" if p_adj < 0.05 else "ns"
    axes[i].set_title(f"{col}  {sig}", fontsize=10)
    axes[i].set_xlabel("")

for j in range(len(TOPO_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Topological features: Alzheimer-simulated vs. Healthy-simulated text", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 4. Emotional profile

Emotion z-scores against the NRC baseline, visualized as a Plutchik flower per group.

![Emotional profile of the healthy-simulated corpus](assets/plutchik_flower_healthy.png)

*Healthy-simulated text shows a clearly differentiated profile: strong positive
emotions (joy, anticipation, trust) well above baseline, negative emotions well
below. The AD-simulated corpus follows the same direction but attenuated —
consistent with the affective flattening reported in the clinical literature.*


In [ ]:
corpus_alz = " ".join(df.loc[df["label"] == "alzheimer", "text"])
corpus_healthy = " ".join(df.loc[df["label"] == "healthy", "text"])

emos.draw_statistically_significant_emotions(corpus_alz)


In [ ]:
emos.draw_statistically_significant_emotions(corpus_healthy)


Below are the aggregated word-association networks for each group (top-degree
words only, for readability). Note how the healthy-simulated network centers on
richly descriptive, sensory vocabulary (aroma, symphony, vibrant), while the
AD-simulated network centers on repetition and word-finding difficulty (forget,
smell, maybe).

![Alzheimer-simulated word network](assets/network_diagram_alzheimer.png)

![Healthy-simulated word network](assets/network_diagram_healthy.png)


## 5. Classification: can these features separate the groups?

Five classifiers are compared on the full feature set (topological + emotional),
under 10-fold stratified cross-validation. Train-test AUC gap is reported alongside
test AUC, as a first check for overfitting.


In [ ]:
def evaluate_classifiers(X, y, label=""):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    classifiers = {
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.1, penalty="l2", max_iter=1000, random_state=42))
        ]),
        "DecisionTree": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42))
        ]),
        "RandomForest": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(n_estimators=200, max_depth=4,
                                            min_samples_leaf=8, max_features="sqrt", random_state=42))
        ]),
        "GradientBoosting": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                                learning_rate=0.1, subsample=0.8, random_state=42))
        ]),
        "XGBoost": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                                       subsample=0.8, colsample_bytree=0.8,
                                       eval_metric="logloss", random_state=42))
        ]),
    }

    rows = []
    for name, clf in classifiers.items():
        y_prob = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:, 1]
        auc_test = roc_auc_score(y, y_prob)
        acc_test = accuracy_score(y, (y_prob >= 0.5).astype(int))

        cv_res = cross_validate(clf, X, y, cv=cv, scoring="roc_auc", return_train_score=True)
        auc_train = cv_res["train_score"].mean()

        rows.append({"Model": name, "AUC_test": round(auc_test, 4),
                     "AUC_train": round(auc_train, 4),
                     "Gap (train-test)": round(auc_train - auc_test, 4),
                     "Accuracy": round(acc_test, 4)})

    return pd.DataFrame(rows).sort_values("AUC_test", ascending=False)


y = LabelEncoder().fit_transform(df_features["label"])  # alzheimer=0, healthy=1

print("Full feature set (topological + emotional):")
evaluate_classifiers(df_features.drop(columns=["label"]).values, y)


The primary reported result (matching the paper) uses emotion features only,
on the reasoning that emotional tone should be less tied to raw text length than
network size metrics are. Logistic regression and decision tree are the two
classifiers we report as the main result; the rest are here as an extended
robustness comparison.


In [ ]:
print("Emotion features only (the 'safe', length-independent feature set):")
evaluate_classifiers(df_features[EMOTION_COLS].values, y)


## 6. SHAP: which features actually drive the classification?

![SHAP summary, full feature set](assets/shap_summary_full.png)

*Network size features (`n_edges`, `n_nodes`, `lcc_size`, `avg_path_length`)
dominate the model's decisions — far more than any single emotion. This is the
first hint that the classifier may be leaning on document length/size rather than
emotional content, even when emotion features are included.*

![SHAP summary, topological features only](assets/shap_intensive_features.png)

*Restricting to intensive (length-independent) topological features, `avg_path_length`,
`density`, and `mean_degree` still separate the classes cleanly — showing the
structural difference is real, but also reinforcing that these features can't
fully escape a relationship with how much text there is to build a network from.*


In [ ]:
X_full = df_features.drop(columns=["label"]).values
feature_cols = [c for c in df_features.columns if c != "label"]

pipe = Pipeline([("scaler", StandardScaler()),
                 ("clf", xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                                            subsample=0.8, colsample_bytree=0.8,
                                            eval_metric="logloss", random_state=42))])
pipe.fit(X_full, y)

X_scaled = pipe.named_steps["scaler"].transform(X_full)
explainer = shap.TreeExplainer(pipe.named_steps["clf"])
shap_values = explainer.shap_values(X_scaled)

shap.summary_plot(shap_values, X_scaled, feature_names=feature_cols)


## 7. Checking for a spurious separation: LDA and PCA

If the classifier's accuracy looks unusually high for a small, LLM-generated
dataset, that's worth checking rather than reporting as-is. LDA gives a single
axis of maximum separation between the labeled groups; PCA checks whether that
separation shows up even without using the labels at all.

![2D PCA of emotion features](assets/pca_scatter.png)

*A near-total, low-dimensional separation between groups on unsupervised structure
alone is a signal that the two simulated groups are unusually homogeneous
internally — not confirmation that the features capture a genuine clinical
marker. See the sanity checks below for a direct test of this.*


In [ ]:
X_emotion_scaled = StandardScaler().fit_transform(df_features[EMOTION_COLS].values)

lda = LinearDiscriminantAnalysis(n_components=1)
lda_scores = lda.fit_transform(X_emotion_scaled, y)

pca = PCA(n_components=2)
pca_scores = pca.fit_transform(X_emotion_scaled)

mpl.rcParams.update({"font.size": 12})
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for label_value, label_name, color in [(0, "Alzheimer", "#e74c3c"), (1, "Healthy", "#2ecc71")]:
    mask = y == label_value
    axes[0].hist(lda_scores[mask, 0], bins=20, alpha=0.6, label=label_name, color=color)
    axes[1].scatter(pca_scores[mask, 0], pca_scores[mask, 1], alpha=0.6, label=label_name, color=color)

axes[0].set_xlabel("LDA Score")
axes[0].set_ylabel("Count")
axes[0].set_title("LDA Projection")
axes[0].legend()

axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].set_title("2D PCA")
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Sanity checks: is the separation a surface-level artifact?

These checks test the LDA/PCA concern directly, at the text level rather than the
feature level. If raw TF-IDF alone separates the groups almost perfectly, or if
document length alone does, that's strong evidence the LLM encoded the class
label into surface style — not that the emotional/network features found a subtle
clinical signal.


In [ ]:
# Sanity check 1: TF-IDF + logistic regression directly on the raw text.
# An AUC near 1.0 here means the two classes are separable at the surface lexical
# level -- i.e. the LLM generated two disjoint writing styles, not two overlapping
# clinical distributions.
texts = df["text"].values
y_raw = LabelEncoder().fit_transform(df["label"])
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

pipe_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
    ("clf", LogisticRegression(C=1.0, max_iter=2000, random_state=42)),
])
auc_tfidf = cross_val_score(pipe_tfidf, texts, y_raw, cv=cv, scoring="roc_auc")
print(f"TF-IDF (1-2 gram) + LogReg   AUC = {auc_tfidf.mean():.4f} +/- {auc_tfidf.std():.4f}")

pipe_tfidf_uni = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 1), min_df=2, max_features=200)),
    ("clf", LogisticRegression(C=1.0, max_iter=2000, random_state=42)),
])
auc_uni = cross_val_score(pipe_tfidf_uni, texts, y_raw, cv=cv, scoring="roc_auc")
print(f"TF-IDF (unigrams, top 200)   AUC = {auc_uni.mean():.4f} +/- {auc_uni.std():.4f}")


In [ ]:
# Sanity check 2: which individual tokens drive that separation?
# Seeing generic AD-narrative markers (forget, thing) on one side and vivid,
# sensory vocabulary (vibrant, aroma, delightful) on the other confirms the LLM
# encoded the class distinction directly into word choice.
pipe_tfidf_uni.fit(texts, y_raw)
vectorizer = pipe_tfidf_uni.named_steps["tfidf"]
classifier = pipe_tfidf_uni.named_steps["clf"]
coefs = pd.Series(classifier.coef_[0], index=vectorizer.get_feature_names_out()).sort_values()

print("Top 15 tokens -> healthy:")
print(coefs.tail(15)[::-1].to_string())
print("\nTop 15 tokens -> alzheimer:")
print(coefs.head(15).to_string())


In [ ]:
# Sanity check 3: basic stylometric features (length, lexical diversity, fillers).
STOPWORDS = set('''
a an the and or but if then so of in on at to for with by from as is are was were be been being
have has had do does did i you he she it we they me him her us them my your his their our this
that these those there here what when where why how which who whom not no yes oh ah um uh hmm
'''.split())

FILLER_RE = re.compile(r"\b(um|uh|hmm|oh|ah|er|like|you know|i mean)\b", re.IGNORECASE)
ELLIPSIS_RE = re.compile(r"(\.{2,}|--|-)")


def stylometry(text):
    words = re.findall(r"\b[\w']+\b", text.lower())
    n_words = len(words)
    content_words = [w for w in words if w not in STOPWORDS]
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]

    return {
        "n_chars": len(text),
        "n_words": n_words,
        "n_sentences": len(sentences),
        "mean_sent_len": n_words / max(len(sentences), 1),
        "ttr": len(set(words)) / max(n_words, 1),
        "content_ratio": len(content_words) / max(n_words, 1),
        "fillers_per100": 100 * len(FILLER_RE.findall(text)) / max(n_words, 1),
        "ellipsis_per100": 100 * len(ELLIPSIS_RE.findall(text)) / max(n_words, 1),
    }


style_df = pd.DataFrame([stylometry(t) for t in texts])
style_df["label"] = df["label"].values
style_df.groupby("label").agg(["mean", "std"]).round(3)


In [ ]:
# Sanity check 4: how well does document length ALONE classify the groups?
# If a single surface statistic like word count already separates the classes
# almost perfectly, that's the clearest possible evidence the difference is
# stylistic, not an emotional or cognitive pattern.
from sklearn.metrics import roc_auc_score as _roc_auc_score

for feature_name in ["n_words", "ttr", "fillers_per100", "ellipsis_per100", "mean_sent_len"]:
    auc = _roc_auc_score(y_raw, style_df[feature_name])
    auc = max(auc, 1 - auc)  # symmetrize: direction of the relationship isn't the point here
    print(f"AUC using only '{feature_name}': {auc:.4f}")


## Notes on interpreting the results

The topological and emotional differences line up with what's reported in the
clinical literature on dementia and language: smaller, more fragmented semantic
networks, and a flattened emotional profile. That part of the analysis holds up.

The classification result is where this notebook earns its "exploratory" label.
Both the LDA/PCA check and the sanity checks above point the same direction: this
particular synthetic corpus is probably too internally homogeneous, and too
cleanly different between groups, for the classification AUC to be read as a
validated clinical signal. See `docs/report.pdf` for the full discussion,
including why real transcript data (DementiaBank Pitt Corpus, ADReSS challenge)
is the natural next step before treating any of this as evidence the method
detects Alzheimer's disease from text in general.
